# Automated FinTech Portfolio Analysis Project

This project fetches stock market data using Python and prepares data for portfolio analysis and dashboard visualization.

!pip install yfinance  #yfinance is a finance data specific python library that will fetch data from yahoo finance site

In [1]:
import yfinance as yf 
import pandas as pd

In [2]:
stock_dictionary = {
    "Finance": ["JPM", "MS", "GS", "BAC", "C"],
    "Technology": ["AAPL", "MSFT", "AMZN", "GOOG", "NVDA"],
    "Healthcare": ["PFE", "JNJ", "MRK", "UNH", "ABBV"],
    "Automobile": ["TSLA", "F", "GM", "TM", "RIVN","TATAMOTORS.NS"]
}
tickers = []

for stocks in stock_dictionary.values():
    tickers.extend(stocks)

try:
    portfolio_data = yf.download(tickers, period="6mo")

except Exception as e:
    print("Download failed")
    print(e)

[**********************90%******************     ]  19 of 21 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  21 of 21 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


In [3]:
portfolio_data.shape

(124, 106)

In [4]:
portfolio_data.head(5)

Price          Adj Close       Close                                     \
Ticker     TATAMOTORS.NS        AAPL        ABBV        AMZN        BAC   
Date                                                                      
2025-12-19           NaN  273.162506  223.158279  227.350006  54.675228   
2025-12-22           NaN  270.467499  224.230682  228.429993  55.278664   
2025-12-23           NaN  271.854919  225.096466  232.139999  55.367695   
2025-12-24           NaN  273.302216  226.178726  232.380005  55.644680   
2025-12-26           NaN  272.892975  226.267242  232.520004  55.565536   

Price                                                                 ...  \
Ticker               C          F         GM        GOOG          GS  ...   
Date                                                                  ...   
2025-12-19  113.726440  13.159799  81.977776  308.207245  884.902466  ...   
2025-12-22  116.924553  13.150029  82.654701  310.923676  890.369507  ...   
2025-12-23  118.221634  12.983944  82.375969  315.268005  893.053589  ...   
2025-12-24  120.360313  13.052332  82.505379  315.258026  902.036438  ...   
2025-12-26  119.231567  13.003483  82.684563  314.548950  898.332275  ...   

Price         Volume                                                      \
Ticker           MRK        MS      MSFT       NVDA       PFE       RIVN   
Date                                                                       
2025-12-19  44980200  10639400  70836100  324925900  87590000  106165100   
2025-12-22  17136200   4574600  16963000  129064400  38758900   38448500   
2025-12-23  13587500   3514800  14683600  174873600  43692900   38763700   
2025-12-24   5335000   2648000   5855900   65528500  19328500   10629600   
2025-12-26   6280700   2564700   8842200  139740300  21617800   20503000   

Price                                                  
Ticker     TATAMOTORS.NS      TM       TSLA       UNH  
Date                                                   
2025-12-19           NaN  533700  103305400  10553000  
2025-12-22           NaN  238400   86916100   7872400  
2025-12-23           NaN  330500   58223600   4463300  
2025-12-24           NaN  170800   41285400   2842700  
2025-12-26           NaN  146000   58780700   4359300  

[5 rows x 106 columns]

In [5]:
# data quality checks
failed_tickers = portfolio_data["Close"].isna().all()
failed_tickers = failed_tickers[failed_tickers]
failed_ticker_list = failed_tickers.index.tolist()
print("Failed tickers:")
print(failed_ticker_list)

Failed tickers:
['TATAMOTORS.NS']


In [6]:
portfolio_data = portfolio_data.drop(
    columns=failed_ticker_list,
    level=1
)

#### Clean dataframe presentation

In [7]:
latest_date = portfolio_data.index[-1]

In [8]:
clean_table = pd.DataFrame({
    "Close": portfolio_data["Close"].loc[latest_date],
    "High": portfolio_data["High"].loc[latest_date],
    "Low": portfolio_data["Low"].loc[latest_date],
    "Open": portfolio_data["Open"].loc[latest_date],
    "Volume": portfolio_data["Volume"].loc[latest_date]
})

In [9]:
clean_table.head()

,Close,High,Low,Open,Volume
Ticker,,,,,
AAPL,298.010010,300.570007,295.619995,298.109985,85962200
ABBV,216.490005,222.350006,215.369995,221.369995,9646100
AMZN,244.389999,245.729996,236.020004,240.119995,75624400
BAC,56.200001,57.330002,56.029999,57.230000,70438800
C,143.059998,147.960007,143.039993,146.490005,29331200


In [10]:
# need date column as well
clean_table = clean_table.reset_index()

In [11]:
clean_table.head(1)

,Ticker,Close,High,Low,Open,Volume
0,AAPL,298.01001,300.570007,295.619995,298.109985,85962200


In [12]:
# add date column
clean_table["Date"] = latest_date

In [13]:
# stock data of lastest date
clean_portfolio_data = clean_table[["Ticker","Date","Close","High","Low","Open","Volume"]]
clean_portfolio_data.head()

,Ticker,Date,Close,High,Low,Open,Volume
0,AAPL,2026-06-18,298.010010,300.570007,295.619995,298.109985,85962200
1,ABBV,2026-06-18,216.490005,222.350006,215.369995,221.369995,9646100
2,AMZN,2026-06-18,244.389999,245.729996,236.020004,240.119995,75624400
3,BAC,2026-06-18,56.200001,57.330002,56.029999,57.230000,70438800
4,C,2026-06-18,143.059998,147.960007,143.039993,146.490005,29331200


In [14]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [15]:
clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)

C:\Users\Ashlesha\AppData\Local\Temp\ipykernel_34480\1232821456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)


In [16]:
clean_portfolio_data = clean_portfolio_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

clean_portfolio_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2026-06-18,298.010010,300.570007,295.619995,298.109985,85962200
1,ABBV,Healthcare,2026-06-18,216.490005,222.350006,215.369995,221.369995,9646100
2,AMZN,Technology,2026-06-18,244.389999,245.729996,236.020004,240.119995,75624400
3,BAC,Finance,2026-06-18,56.200001,57.330002,56.029999,57.230000,70438800
4,C,Finance,2026-06-18,143.059998,147.960007,143.039993,146.490005,29331200


In [17]:
clean_portfolio_data.isnull().sum()

Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [18]:
clean_portfolio_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  20 non-null     object        
 1   Sector  20 non-null     object        
 2   Date    20 non-null     datetime64[ns]
 3   Close   20 non-null     float64       
 4   High    20 non-null     float64       
 5   Low     20 non-null     float64       
 6   Open    20 non-null     float64       
 7   Volume  20 non-null     int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 1.4+ KB


In [19]:
clean_portfolio_data['Date'] = pd.to_datetime(clean_portfolio_data['Date'],format='%d-%m-%Y')

In [20]:
clean_portfolio_data.describe()

,Date,Close,High,Low,Open,Volume
count,20,20.000000,20.000000,20.000000,20.000000,2.000000e+01
mean,2026-06-18 00:00:00,250.669002,254.890801,247.656398,252.804251,5.080312e+07
min,2026-06-18 00:00:00,14.060000,14.130000,13.720000,14.120000,4.594000e+05
25%,2026-06-18 00:00:00,105.225002,106.442501,103.412500,106.132500,1.715898e+07
50%,2026-06-18 00:00:00,219.830002,226.410004,219.129997,225.229996,2.943035e+07
75%,2026-06-18 00:00:00,335.779999,345.817497,332.272003,343.685013,7.173520e+07
max,2026-06-18 00:00:00,1096.560059,1125.000000,1093.329956,1120.000000,2.412720e+08
std,NaN,237.048297,242.567408,235.606374,241.265876,5.569330e+07


In [21]:
# check duplicates
clean_portfolio_data.duplicated().sum()

0

In [22]:
# clean dataset for the latest date
portfolio_data_with_sectors_lastestDay = clean_portfolio_data.copy()

In [23]:
portfolio_data.head(2)

Price            Close                                                 \
Ticker            AAPL        ABBV        AMZN        BAC           C   
Date                                                                    
2025-12-19  273.162506  223.158279  227.350006  54.675228  113.726440   
2025-12-22  270.467499  224.230682  228.429993  55.278664  116.924553   

Price                                                                 ...  \
Ticker              F         GM        GOOG          GS         JNJ  ...   
Date                                                                  ...   
2025-12-19  13.159799  81.977776  308.207245  884.902466  204.104904  ...   
2025-12-22  13.150029  82.654701  310.923676  890.369507  205.044464  ...   

Price         Volume                                                     \
Ticker           JPM       MRK        MS      MSFT       NVDA       PFE   
Date                                                                      
2025-12-19  24494400  44980200  10639400  70836100  324925900  87590000   
2025-12-22   8354600  17136200   4574600  16963000  129064400  38758900   

Price                                               
Ticker           RIVN      TM       TSLA       UNH  
Date                                                
2025-12-19  106165100  533700  103305400  10553000  
2025-12-22   38448500  238400   86916100   7872400  

[2 rows x 100 columns]

In [24]:
historical_stock_data = portfolio_data.copy()

In [25]:
historical_stock_data = (historical_stock_data.stack(level=1).reset_index())

historical_stock_data.head()

Price,Date,Ticker,Close,High,Low,Open,Volume
0,2025-12-19,AAPL,273.162506,274.090774,269.399478,271.645305,144632000
1,2025-12-19,ABBV,223.158279,225.765492,218.760441,219.350746,18968800
2,2025-12-19,AMZN,227.350006,229.130005,225.580002,226.759995,85544400
3,2025-12-19,BAC,54.675228,54.714799,53.794806,53.933299,73001900
4,2025-12-19,C,113.726440,114.469039,111.934306,112.082819,37660800


In [26]:
historical_stock_data.columns

Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [27]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [28]:
historical_stock_data["Sector"] = historical_stock_data["Ticker"].map(ticker_sector)

In [29]:
historical_stock_data = historical_stock_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

In [30]:
historical_stock_data.head(2)

Price,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000
1,ABBV,Healthcare,2025-12-19,223.158279,225.765492,218.760441,219.350746,18968800


In [31]:
historical_stock_data.isnull().sum()

Price
Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [32]:
historical_stock_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2480 entries, 0 to 2479
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  2480 non-null   object        
 1   Sector  2480 non-null   object        
 2   Date    2480 non-null   datetime64[ns]
 3   Close   2480 non-null   float64       
 4   High    2480 non-null   float64       
 5   Low     2480 non-null   float64       
 6   Open    2480 non-null   float64       
 7   Volume  2480 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 155.1+ KB


In [33]:
historical_stock_data['Date'] = pd.to_datetime(historical_stock_data['Date'],format='%d-%m-%Y')

In [34]:
historical_stock_data.describe()

Price,Date,Close,High,Low,Open,Volume
count,2480,2480.000000,2480.000000,2480.000000,2480.000000,2.480000e+03
mean,2026-03-21 12:23:13.548387072,233.642477,236.463933,230.670088,233.517519,3.133822e+07
min,2025-12-19 00:00:00,11.070457,11.307469,10.971701,11.218589,1.460000e+05
25%,2026-02-04 18:00:00,96.220104,97.349185,95.231920,95.810628,7.199175e+06
50%,2026-03-21 12:00:00,212.401253,214.490013,209.504997,211.340004,1.686015e+07
75%,2026-05-05 06:00:00,305.671967,308.696558,301.707706,305.087502,4.169158e+07
max,2026-06-18 00:00:00,1099.140015,1125.000000,1094.290039,1120.000000,3.608079e+08
std,NaN,200.136432,202.746595,197.317924,199.944872,4.013853e+07


In [35]:
historical_stock_data.duplicated().sum()

0

In [36]:
# cleanups
historical_stock_data.columns.name = None
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [37]:
historical_stock_data["Daily Return %"] = round(historical_stock_data.groupby('Ticker')['Close'].pct_change()*100,2)

In [38]:
historical_stock_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000,NaN
20,AAPL,Technology,2025-12-22,270.467499,273.372106,270.008360,272.353978,36571800,-0.99


In [39]:
historical_stock_data["Daily Return %"].describe()

count    2460.000000
mean        0.053683
std         2.119404
min       -19.610000
25%        -1.050000
50%         0.000000
75%         1.122500
max        26.640000
Name: Daily Return %, dtype: float64

In [40]:
cumulative_returns = historical_stock_data.groupby('Ticker').agg(
                                                                 First_close = ('Close','first'),
                                                                 Last_close = ('Close','last'))
cumulative_returns

,First_close,Last_close
Ticker,,
AAPL,273.162506,298.010010
ABBV,223.158279,216.490005
AMZN,227.350006,244.389999
BAC,54.675228,56.200001
C,113.726440,143.059998
F,13.159799,14.060000
GM,81.977776,79.290001
GOOG,308.207245,367.459991
GS,884.902466,1096.560059


In [41]:
cumulative_returns["Cumulative Return %"] = round((cumulative_returns['Last_close'] - cumulative_returns['First_close']) / cumulative_returns['First_close'] *100 , 2)

cumulative_returns = cumulative_returns.sort_values("Cumulative Return %",ascending=False)
cumulative_returns.head()

,First_close,Last_close,Cumulative Return %
Ticker,,,
MS,175.068893,223.169998,27.48
C,113.726440,143.059998,25.79
UNH,323.049103,400.959991,24.12
GS,884.902466,1096.560059,23.92
GOOG,308.207245,367.459991,19.22


In [42]:
# average trading volume
cumulative_returns["Total Volume"] = (historical_stock_data.groupby("Ticker")["Volume"].sum())

# sector mapping
sector_map = (historical_stock_data.groupby("Ticker")["Sector"].first())
cumulative_returns["Sector"] = sector_map

In [43]:
stock_performance_summary = (cumulative_returns.reset_index())

stock_performance_summary = stock_performance_summary[["Ticker","Sector","First_close","Last_close","Cumulative Return %","Total Volume"]]
stock_performance_summary.head()

,Ticker,Sector,First_close,Last_close,Cumulative Return %,Total Volume
0,MS,Finance,175.068893,223.169998,27.48,817812900
1,C,Finance,113.726440,143.059998,25.79,1710036400
2,UNH,Healthcare,323.049103,400.959991,24.12,1069860400
3,GS,Finance,884.902466,1096.560059,23.92,283465200
4,GOOG,Technology,308.207245,367.459991,19.22,2581921500


In [44]:
# investor's porfolios 

investor_profiles = { 
    "Investor_A": {"capital": 10000, "weights": {"JPM": 0.30, "BAC": 0.30, "ABBV": 0.20, "JNJ": 0.20}},
    "Investor_B": { "capital": 10000,"weights": {"NVDA": 0.30, "TSLA": 0.25, "AMZN": 0.25, "AAPL": 0.20}}
}
investor_profiles

{'Investor_A': {'capital': 10000,
  'weights': {'JPM': 0.3, 'BAC': 0.3, 'ABBV': 0.2, 'JNJ': 0.2}},
 'Investor_B': {'capital': 10000,
  'weights': {'NVDA': 0.3, 'TSLA': 0.25, 'AMZN': 0.25, 'AAPL': 0.2}}}

In [45]:
for investor, details in investor_profiles.items():

    total_weight = sum(details["weights"].values())

    if total_weight != 1:
        raise ValueError(
            f"{investor} weights do not sum to 1. Current total: {total_weight}"
        )

-  validated that portfolio allocations summed to 100%, if not, the pipeline stops and raises an error to prevent incorrect return calculations

#### now lets calculate weighted return calculation for each investor

##### Investor A portfolio summary

In [46]:
investor_A_stocks = investor_profiles['Investor_A']['weights'].keys()
investor_A_stocks

dict_keys(['JPM', 'BAC', 'ABBV', 'JNJ'])

In [47]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_A_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_A_stocks)].copy()
investor_A_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
1,ABBV,Healthcare,2025-12-19,223.158279,225.765492,218.760441,219.350746,18968800,NaN
21,ABBV,Healthcare,2025-12-22,224.230682,224.958726,220.629765,222.459738,4828100,0.48
41,ABBV,Healthcare,2025-12-23,225.096466,227.064179,224.230683,224.535676,3612100,0.39
61,ABBV,Healthcare,2025-12-24,226.178726,227.074039,225.283413,225.755675,1744900,0.48
81,ABBV,Healthcare,2025-12-26,226.267242,226.887077,224.988226,226.109837,1593900,0.04


In [48]:
investor_A_data['Weight'] = investor_A_data['Ticker'].map(investor_profiles['Investor_A']['weights'])

In [49]:
investor_A_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
1,ABBV,Healthcare,2025-12-19,223.158279,225.765492,218.760441,219.350746,18968800,NaN,0.2
21,ABBV,Healthcare,2025-12-22,224.230682,224.958726,220.629765,222.459738,4828100,0.48,0.2


In [50]:
investor_A_data["Weighted Return"] = (investor_A_data["Daily Return %"] * investor_A_data["Weight"])

In [51]:
portfolio_A_daily_returns = (investor_A_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_A_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [52]:
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-19,0.000
1,2025-12-22,1.073
2,2025-12-23,0.242
3,2025-12-24,0.737
4,2025-12-26,-0.162


In [53]:
initial_capital = investor_profiles["Investor_A"]["capital"]

portfolio_A_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_A_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-19,0.000,10000.000
1,2025-12-22,1.073,10107.300
2,2025-12-23,0.242,10131.760
3,2025-12-24,0.737,10206.431
4,2025-12-26,-0.162,10189.896


In [54]:
portfolio_A_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_A_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_A_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-19,0.000,10000.00,0.000
1,2025-12-22,1.073,10107.30,1.073
2,2025-12-23,0.242,10131.76,1.318


##### Investor B portfolio summary

In [55]:
investor_B_stocks = investor_profiles['Investor_B']['weights'].keys()
investor_B_stocks

dict_keys(['NVDA', 'TSLA', 'AMZN', 'AAPL'])

In [56]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_B_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_B_stocks)].copy()
investor_B_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000,NaN
20,AAPL,Technology,2025-12-22,270.467499,273.372106,270.008360,272.353978,36571800,-0.99
40,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000,0.51
60,AAPL,Technology,2025-12-24,273.302216,274.919206,271.695216,271.834940,17910600,0.53
80,AAPL,Technology,2025-12-26,272.892975,274.859323,272.353968,273.651575,21521800,-0.15


In [57]:
investor_B_data['Weight'] = investor_B_data['Ticker'].map(investor_profiles['Investor_B']['weights'])

In [58]:
investor_B_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
0,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000,NaN,0.2
20,AAPL,Technology,2025-12-22,270.467499,273.372106,270.008360,272.353978,36571800,-0.99,0.2


In [59]:
investor_B_data["Weighted Return"] = (investor_B_data["Daily Return %"] * investor_B_data["Weight"])

In [60]:
portfolio_B_daily_returns = (investor_B_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_B_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [61]:
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-19,0.0000
1,2025-12-22,0.7590
2,2025-12-23,1.2475
3,2025-12-24,0.0275
4,2025-12-26,-0.2340


In [62]:
initial_capital = investor_profiles["Investor_B"]["capital"]

portfolio_B_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_B_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-19,0.0000,10000.000
1,2025-12-22,0.7590,10075.900
2,2025-12-23,1.2475,10201.597
3,2025-12-24,0.0275,10204.402
4,2025-12-26,-0.2340,10180.524


In [63]:
portfolio_B_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_B_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_B_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-19,0.0000,10000.000,0.000
1,2025-12-22,0.7590,10075.900,0.759
2,2025-12-23,1.2475,10201.597,2.016


In [64]:
portfolio_A_daily_returns.iloc[-1]

Date                             2026-06-18 00:00:00
Portfolio Daily Return %                      -1.839
Portfolio Value                            10437.099
Portfolio Cumulative Return %                  4.371
Name: 123, dtype: object

In [65]:
portfolio_B_daily_returns.iloc[-1]

Date                             2026-06-18 00:00:00
Portfolio Daily Return %                        2.01
Portfolio Value                            10503.972
Portfolio Cumulative Return %                   5.04
Name: 123, dtype: object

In [66]:
return_A = portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1]
return_B = portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1]

In [67]:
# expected annual return is

annual_return_A = round((portfolio_A_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
annual_return_B = round((portfolio_B_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
print(annual_return_A)
print(annual_return_B)

9.83
12.93


#### calculate risk to check which portfolio is more volatile

In [68]:
portfolio_A_daily_returns["Portfolio Daily Return %"].std()

0.9560479923296505

In [69]:
portfolio_B_daily_returns["Portfolio Daily Return %"].std()

1.531701519903902

In [70]:
annual_risk_A = (portfolio_A_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

annual_risk_B = (portfolio_B_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

print(round(annual_risk_A,2))
print(round(annual_risk_B,2))

15.18
24.32


In [71]:
# assuming risk-free rate as 0 
sharpe_A = round(annual_return_A / annual_risk_A,2)
sharpe_B = round(annual_return_B / annual_risk_B,2)
print(sharpe_A)
print(sharpe_B)

0.65
0.53


In [72]:
# creating the overall summary
portfolio_summary = pd.DataFrame({
                                "Portfolio": ["Conservative", "Growth"],
                                "Capital": [10000, 10000],
                                "Current Value": [
                                        round(portfolio_A_daily_returns["Portfolio Value"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Value"].iloc[-1], 2)],
                                "Total Return %": [
                                        round(portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2)],
                                "Annual Return %": [
                                        annual_return_A,
                                        annual_return_B],
                                "Annual Risk %": [
                                        round(annual_risk_A, 2),
                                        round(annual_risk_B, 2)],
                                "Sharpe Ratio": [
                                        sharpe_A,
                                        sharpe_B]
                                })

In [73]:
portfolio_summary.head()

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10437.10,4.37,9.83,15.18,0.65
1,Growth,10000,10503.97,5.04,12.93,24.32,0.53


In [74]:
portfolio_A_daily_returns["Portfolio"] = "Conservative"

In [75]:
portfolio_B_daily_returns["Portfolio"] = "Growth"

In [76]:
portfolio_daily_returns = pd.concat([portfolio_A_daily_returns, portfolio_B_daily_returns],ignore_index=True)

In [77]:
portfolio_daily_returns = portfolio_daily_returns[["Date","Portfolio","Portfolio Daily Return %","Portfolio Value",
                                                   "Portfolio Cumulative Return %"]]

In [78]:
portfolio_daily_returns = portfolio_daily_returns.sort_values(["Date", "Portfolio"])

In [79]:
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)

In [80]:
portfolio_daily_returns.head()

,Date,Portfolio,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-19,Conservative,0.000,10000.00,0.000
1,2025-12-19,Growth,0.000,10000.00,0.000
2,2025-12-22,Conservative,1.073,10107.30,1.073
3,2025-12-22,Growth,0.759,10075.90,0.759
4,2025-12-23,Conservative,0.242,10131.76,1.318


In [81]:
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [82]:
historical_stock_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000,NaN
20,AAPL,Technology,2025-12-22,270.467499,273.372106,270.008360,272.353978,36571800,-0.99
40,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000,0.51
60,AAPL,Technology,2025-12-24,273.302216,274.919206,271.695216,271.834940,17910600,0.53
80,AAPL,Technology,2025-12-26,272.892975,274.859323,272.353968,273.651575,21521800,-0.15


In [83]:
print(historical_stock_data.shape)
print(portfolio_daily_returns.shape)
print(portfolio_summary.shape)

(2480, 9)
(248, 5)
(2, 7)


In [84]:
historical_stock_data = historical_stock_data.reset_index(drop=True)
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)
portfolio_summary = portfolio_summary.reset_index(drop=True)

In [85]:
historical_stock_data.to_csv("historical_stock_data.csv", index=False)
portfolio_daily_returns.to_csv("portfolio_daily_returns.csv", index=False)
portfolio_summary.to_csv("portfolio_summary.csv", index=False)
stock_performance_summary.to_csv("stock_performance_summary.csv",index=False)

### generate PDF report

In [86]:
import pandas as pd

In [87]:
portfolio_summary = pd.read_csv("portfolio_summary.csv")

portfolio_summary

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10437.10,4.37,9.83,15.18,0.65
1,Growth,10000,10503.97,5.04,12.93,24.32,0.53


In [88]:
max_return  = portfolio_summary["Total Return %"].max()
max_return

5.04

In [89]:
best_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"]==max_return,"Portfolio"].iloc[0]
worst_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"] == portfolio_summary["Total Return %"].min(),"Portfolio"].iloc[0]

In [90]:
print(best_portfolio)
print(worst_portfolio)

Growth
Conservative


In [91]:
return_difference = round(portfolio_summary["Total Return %"].max() - portfolio_summary["Total Return %"].min(),2) 

In [92]:
return_difference

0.67

In [93]:
print(f"Best Performing Portfolio: {best_portfolio}")
print(f"{best_portfolio} Portfolio Outperformed {worst_portfolio} Portfolio by {return_difference}%")

Best Performing Portfolio: Growth
Growth Portfolio Outperformed Conservative Portfolio by 0.67%


In [94]:
portfolio_daily_returns = pd.read_csv("portfolio_daily_returns.csv")

In [95]:
report_date = portfolio_daily_returns["Date"].max()
report_date

'2026-06-18'

In [96]:
report_text = f"""
📊 Portfolio Report

Report Date: {report_date}

"""

In [97]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-18




In [98]:
for _, portfolio in portfolio_summary.iterrows():

    report_text += f"""

Portfolio: {portfolio["Portfolio"]}

Current Value: ₹{portfolio["Current Value"]}

Total Return: {portfolio["Total Return %"]}%

Annual Return: {portfolio["Annual Return %"]}%

Annual Risk: {portfolio["Annual Risk %"]}%

Sharpe Ratio: {portfolio["Sharpe Ratio"]}

"""

In [99]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-18



Portfolio: Conservative

Current Value: ₹10437.1

Total Return: 4.37%

Annual Return: 9.83%

Annual Risk: 15.18%

Sharpe Ratio: 0.65



Portfolio: Growth

Current Value: ₹10503.97

Total Return: 5.04%

Annual Return: 12.93%

Annual Risk: 24.32%

Sharpe Ratio: 0.53




In [100]:
report_text += f"""

Portfolio Comparison

Best Performing Portfolio: {best_portfolio}

{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%

"""

In [101]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-18



Portfolio: Conservative

Current Value: ₹10437.1

Total Return: 4.37%

Annual Return: 9.83%

Annual Risk: 15.18%

Sharpe Ratio: 0.65



Portfolio: Growth

Current Value: ₹10503.97

Total Return: 5.04%

Annual Return: 12.93%

Annual Risk: 24.32%

Sharpe Ratio: 0.53



Portfolio Comparison

Best Performing Portfolio: Growth

Growth Portfolio Outperformed Conservative Portfolio by 0.67%




!pip install reportlab

In [102]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [103]:
# creating a variable to save the path for report on my system
pdf_path = f"reports/portfolio_report_{report_date}.pdf"

In [104]:
# create an empty pdf document object 
doc = SimpleDocTemplate(pdf_path)

In [105]:
# get default text formatting styles
styles = getSampleStyleSheet()

In [106]:
elements = []

In [107]:
# add Title and recent date
elements.append(Paragraph("Portfolio Performance Report", styles["Title"]))

elements.append(Paragraph(f"Report Date: {report_date}", styles["Normal"]))

elements.append(Spacer(1, 12))

In [108]:
# add portfolio details of each
for _, portfolio in portfolio_summary.iterrows():

    elements.append(Paragraph(f"<b>Portfolio:</b> {portfolio['Portfolio']}",styles["Heading2"]))

    elements.append(Paragraph(f"Current Value: ₹{portfolio['Current Value']}",styles["Normal"]))

    elements.append(Paragraph(f"Total Return: {portfolio['Total Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Return: {portfolio['Annual Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Risk: {portfolio['Annual Risk %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Sharpe Ratio: {portfolio['Sharpe Ratio']}",styles["Normal"]))

    elements.append(Spacer(1, 12))

In [109]:
# add the comparision section
elements.append(Paragraph("Portfolio Comparison", styles["Heading1"]))

elements.append(Paragraph(f"Best Performing Portfolio: {best_portfolio}",styles["Normal"]))

elements.append(Paragraph(f"{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%",styles["Normal"]))

In [110]:
print(len(elements))

20


In [111]:
doc.build(elements)

print("PDF Report Created Successfully!")

PDF Report Created Successfully!
